In [1]:
import pandas as pd
import readability
import os
import numpy as np
import random
import torch
from bert_score import BERTScorer
import re
import nltk
from nltk.translate import meteor
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
import math

In [2]:
from transformers import logging
logging.set_verbosity_error() # make sure only important transformers logging output is visible

In [3]:
INPUT_DIR = 'syntheticNotesLocal'
OUTPUT_DIR = 'metricResults'

In [4]:
# Only need to run once.
# nltk.download('all')

In [5]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [6]:
def preprocess_text(text):
    text = re.sub(r'[^\w\s/]', '', text)
    # tokenize
    tokens = word_tokenize(text.lower())
    # remove stop words
    filtered_tokens = [token for token in tokens if token not in stopwords.words('english')]
    # lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    # join the tokens back into a string
    processed_text = ' '.join(lemmatized_tokens)
    return processed_text

In [7]:
fake_notes = pd.read_excel(f'./{INPUT_DIR}/fakeNoteGeneration/fake_notes.xlsx')
fake_readability = readability.getmeasures(('\n'.join(fake_notes['Note'].tolist())), lang='en')['readability grades']['SMOGIndex']
real_notes = pd.read_excel(f'./{INPUT_DIR}/realNoteGeneration/real_notes.xlsx')
real_readability = readability.getmeasures(('\n'.join(real_notes['Note'].tolist())), lang='en')['readability grades']['SMOGIndex']

In [8]:
print(f"Fake Note Readability: {fake_readability}")
print(f"Real Note Readability: {real_readability}")

Fake Note Readability: 16.74772708486752
Real Note Readability: 15.36931687685298


In [9]:
def make_scores(reference_readability, file_path, notes_df_used_in_generation):
    average_scores_dict = []
    # Make BERT scorer.
    scorer = BERTScorer(model_type="bert-base-uncased")
    for file in os.listdir(f"./{file_path}"):
        if file.endswith('.csv'):
            temp_df = pd.read_csv(f"./{file_path}/{file}")
            file_name = file.split('_')
            index, needs = int(file_name[0]), file_name[1].replace('.csv', '')
            # --------- Readability (number linked to grade of reading level)s ---------
            references = []
            candidates = []
            meteors = []
            abs_sentiment_diffs = []
            abs_subjectivity_diffs = []
            # Make readability metric.
            text_series = temp_df['report'].tolist()
            long_text = '\n'.join(text_series)
            candidate_readability = readability.getmeasures(long_text, lang='en')['readability grades']['SMOGIndex']
            abs_readability_diff = (math.sqrt((reference_readability - candidate_readability) ** 2))
            temp_prompt = notes_df_used_in_generation[notes_df_used_in_generation['Needs'] == needs]['Note'].tolist()[index]
            preprocessed_prompt = preprocess_text(temp_prompt)
            reference_blob = TextBlob(preprocessed_prompt)
            for temp_generation in temp_df['report'].tolist():
                processed_generation = preprocess_text(temp_generation)
                candidate_blob = TextBlob(processed_generation)
                if candidate_blob.sentences and reference_blob.sentences:
                    # Make candidates and references without punctuation for metrics (BERTScore penalizes punctuation, we should take this into account - https://aclanthology.org/2023.findings-acl.381.pdf).
                    reference = re.sub(r'[^\w\s/]', '', temp_prompt)
                    candidate = re.sub(r'[^\w\s/]', '', temp_generation)
                    references.append(reference)
                    candidates.append(candidate)
                    # Make METEOR
                    meteors.append(meteor([word_tokenize(candidate)], word_tokenize(reference)))
                    # Make sentiment and subjectivity absolute differences.                    
                    abs_sentiment_diffs.append(math.sqrt((reference_blob.sentences[0].sentiment.polarity - candidate_blob.sentences[0].sentiment.polarity) ** 2))
                    abs_subjectivity_diffs.append(math.sqrt((reference_blob.sentences[0].sentiment.subjectivity - candidate_blob.sentences[0].sentiment.subjectivity) ** 2))

            # Make BERTScore
            _, _, F1 = scorer.score(candidates, references)

            average_scores_dict.append({
                'dataset': file.replace('.csv', ''), 
                'abs_readability_diff': abs_readability_diff,
                # save the F1 values, as these are recommended for use by the BERTScore authors - http://arxiv.org/abs/1904.09675
                'bertscore': float(F1.mean()),
                'meteor': sum(meteors) / len(meteors),
                'abs_sentiment_diff': sum(abs_sentiment_diffs) / len(abs_sentiment_diffs),
                'abs_subjectivity_diff': sum(abs_subjectivity_diffs) / len(abs_subjectivity_diffs)})

    return average_scores_dict

In [10]:
average_real_scores = make_scores(real_readability, f"./{INPUT_DIR}/realNoteGeneration/realNoteSyntheticNotes", real_notes)
average_fake_scores = make_scores(fake_readability, f"./{INPUT_DIR}/fakeNoteGeneration/fakeNoteSyntheticNotes", fake_notes)

/home/isabel/anaconda3/envs/localSyntheticData/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
# min-max normalize every score
real_df = pd.DataFrame(average_real_scores)
real_df['data'] = 'real'
fake_df = pd.DataFrame(average_fake_scores)
fake_df['data'] = 'fake'

resultant_df = pd.concat([real_df, fake_df])

In [12]:
resultant_df

,dataset,abs_readability_diff,bertscore,meteor,abs_sentiment_diff,abs_subjectivity_diff,data
0,0_met,3.834249,0.440531,0.034054,0.170582,0.400842,real
1,0_unmet,6.521434,0.469334,0.054182,0.132170,0.162771,real
2,1_met,5.694357,0.542154,0.097237,0.171102,0.144815,real
3,1_unmet,4.081771,0.466853,0.045491,0.142001,0.195525,real
4,2_met,4.929706,0.489286,0.093466,0.123250,0.167978,real
5,2_unmet,6.328277,0.416064,0.006076,0.103462,0.373097,real
6,3_met,6.130735,0.537005,0.136472,0.135545,0.160060,real
7,3_unmet,4.548684,0.480974,0.056635,0.436368,0.369936,real
8,4_met,4.550569,0.482733,0.117407,0.197961,0.259941,real
9,4_unmet,5.959804,0.480839,0.062092,0.152270,0.196416,real


In [13]:
scaled_df = resultant_df.copy()

metric_cols = [
    "abs_readability_diff",
    "bertscore",
    "meteor",
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]

# Min-max normalize safely.
for col in metric_cols:
    min_val = scaled_df[col].min()
    max_val = scaled_df[col].max()

    if max_val - min_val == 0:
        scaled_df[col] = 0.0
    else:
        scaled_df[col] = (scaled_df[col] - min_val) / (max_val - min_val)

# Invert the lower is better columns.
lower_is_better = [
    "abs_readability_diff",
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]

for col in lower_is_better:
    scaled_df[col] = 1 - scaled_df[col]

# Get overall scores.
scaled_df["overall_score"] = scaled_df[[
    "bertscore",
    "meteor",
    "abs_readability_diff",
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]].mean(axis=1)

mean_scores = scaled_df.groupby('data').mean(numeric_only=True)
mean_scores['data'] = mean_scores.index

# Save outputs. 
os.makedirs(f'./{OUTPUT_DIR}/', exist_ok=True)
scaled_df.to_csv(f'./{OUTPUT_DIR}/evaluated_scaled_results.csv', index=False)
resultant_df.to_csv(f'./{OUTPUT_DIR}/raw_evaluation_results.csv', index=False)
mean_scores.to_csv(f'./{OUTPUT_DIR}/mean_results.csv', index=False)

In [14]:
mean_scores

,abs_readability_diff,bertscore,meteor,abs_sentiment_diff,abs_subjectivity_diff,overall_score,data
data,,,,,,,
fake,0.734362,0.790583,0.715961,0.746697,0.625389,0.722599,fake
real,0.197552,0.497716,0.347451,0.785605,0.680426,0.501750,real
